In [2]:
#Lab task 1

import re
import sys

# =====================================================================
# Part A - Tokenization comparison
# =====================================================================
samples = ["AI, search!", "I can't log in.", "help@example.com",
           "CS4944 version 2.1", "", "یہ ایک مثال ہے۔"]

print("=" * 70)
print("PART A - TOKENIZATION")
print("=" * 70)
for text in samples:
    ws = text.split()
    words = re.findall(r"\b\w+\b", text, flags=re.UNICODE)
    print(repr(text), "whitespace:", ws, len(ws))
    print("word-like:", words, len(words))

assert re.findall(r"\b\w+\b", "") == []

# Model tokenizer comparison for the two English examples.
# The first run downloads the model; if it is unavailable the lexical
# comparison above still stands on its own.
try:
    from sentence_transformers import SentenceTransformer
    _model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",
                                 device="cpu")
    for text in samples[:2]:
        print(text, _model.tokenizer.tokenize(text))
except Exception as error:
    print("Model tokenizer unavailable:", error)

print()
print("Observations:")
print("1. Whitespace splitting keeps punctuation attached to words")
print("   ('search!' stays one token), while the word-like regex drops")
print("   punctuation, splits the contraction \"can't\" into 'can' and")
print("   't', and splits the email address into separate parts.")
print("2. The model tokenizer uses subword units and lowercases, so its")
print("   token list and count differ from both methods above.")
print("Normalization risk: lowercasing or stripping punctuation before")
print("search can damage case-sensitive identifiers (CS4944, file names,")
print("codes) and can remove negation cues, so the original text is")
print("always kept unchanged and any normalized copy is stored separately.")
print("Note: the regex is a simple word-like extractor, not a complete")
print("multilingual tokenizer; the Urdu result shows letter sequences")
print("only and does not prove correct Urdu tokenization.")

# =====================================================================
# Part B - FAQ chatbot
# =====================================================================
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer


class Retriever:
    def __init__(self, records, backend="tfidf"):
        if not records or any(not r["text"].strip() for r in records):
            raise ValueError("Provide nonempty records")
        self.records, self.backend = records, backend
        texts = [r["text"] for r in records]
        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(texts)
        elif backend == "embedding":
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(
                "sentence-transformers/all-MiniLM-L6-v2", device="cpu")
            self.check_length(texts)
            self.matrix = self.model.encode(
                texts, normalize_embeddings=True)
        else:
            raise ValueError("Use tfidf or embedding")

    def check_length(self, texts):
        for text in texts:
            n = len(self.model.tokenizer(text, truncation=False)
                    ["input_ids"])
            if n > self.model.max_seq_length:
                raise ValueError("Input exceeds encoder token limit")

    def search(self, query, k=3, filters=None):
        if not query.strip() or k <= 0:
            return []
        if self.backend == "tfidf":
            q = self.vectorizer.transform([query])
            if q.nnz == 0:
                return []
            scores = (q @ self.matrix.T).toarray()[0]
        else:
            self.check_length([query])
            q = self.model.encode([query], normalize_embeddings=True)[0]
            scores = self.matrix @ q
        allowed = [i for i, r in enumerate(self.records)
                   if all(r.get(key) == value
                          for key, value in (filters or {}).items())]
        ranked = sorted(allowed, key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in ranked[:k]]


FAQ = [
    {"id": "faq:password", "question": "How do I reset my password?",
     "answer": "Use the recovery link on the login page.",
     "source": "it_help.txt"},
    {"id": "faq:submit", "question": "Where do I submit my assignment?",
     "answer": "Use the assignment upload page in the learning portal.",
     "source": "assessment.txt"},
    {"id": "faq:late", "question": "Can illness justify a late submission?",
     "answer": "Request an extension with medical evidence before the deadline.",
     "source": "handbook.txt"},
    # Two student-added FAQs, each with a source.
    {"id": "faq:fees", "question": "How do I pay my semester fee?",
     "answer": "Pay the fee challan at any branch of the designated bank "
               "or through the university online payment portal before the due date.",
     "source": "fees.txt"},
    {"id": "faq:library", "question": "How many books can I borrow from the library?",
     "answer": "Undergraduate students may borrow up to three books "
               "for fourteen days.",
     "source": "library.txt"}]


def build_bot(backend="tfidf"):
    records = [dict(r, text=r["question"] + " " + r["answer"]) for r in FAQ]
    return Retriever(records, backend)


def reply(engine, query):
    if not query.strip():
        return "Please enter a question."
    # Only exact canonical FAQ text is answered automatically here.
    for r in FAQ:
        if query.strip().casefold() == r["question"].casefold():
            return r["answer"] + " [source: " + r["source"] + "]"
    hits = engine.search(query)
    if not hits:
        return "No matching FAQ. Please contact the help desk."
    lines = ["Possible matches; choose or rephrase your question:"]
    for r, score in hits:
        lines.append(r["question"] + f" (similarity {score:.3f})")
    return "\n".join(lines)


if __name__ == "__main__":
    engine = build_bot()
    print()
    print("=" * 70)
    print("PART B - FAQ CHATBOT DEMO (TF-IDF)")
    print("=" * 70)
    for demo_query in ["How do I reset my password?",
                       "How do I pay my semester fee?",
                       "password recovery",
                       "",
                       "volcano geology"]:
        print("You:", demo_query if demo_query else "(empty input)")
        print("Bot:", reply(engine, demo_query))
        print()
    # Behaviour checks
    assert "it_help.txt" in reply(engine, "How do I reset my password?")
    assert reply(engine, "") == "Please enter a question."
    assert "No matching FAQ" in reply(engine, "volcano geology")
    assert "Possible matches" in reply(engine, "password recovery")
    print("FAQ behaviour checks passed")
    print()
    print("Why confirmation? A similarity score alone does not prove a")
    print("stored FAQ answers the question, so non-exact questions return")
    print("candidates for the user to confirm instead of an asserted answer.")

    if "--chat" in sys.argv:
        print()
        print("FAQ assistant. Type exit to stop.")
        while True:
            try:
                query = input("You: ")
            except (EOFError, KeyboardInterrupt):
                break
            if query.strip().lower() == "exit":
                break
            print("Bot:", reply(engine, query))


PART A - TOKENIZATION
'AI, search!' whitespace: ['AI,', 'search!'] 2
word-like: ['AI', 'search'] 2
"I can't log in." whitespace: ['I', "can't", 'log', 'in.'] 4
word-like: ['I', 'can', 't', 'log', 'in'] 5
'help@example.com' whitespace: ['help@example.com'] 1
word-like: ['help', 'example', 'com'] 3
'CS4944 version 2.1' whitespace: ['CS4944', 'version', '2.1'] 3
word-like: ['CS4944', 'version', '2', '1'] 4
'' whitespace: [] 0
word-like: [] 0
'یہ ایک مثال ہے۔' whitespace: ['یہ', 'ایک', 'مثال', 'ہے۔'] 4
word-like: ['یہ', 'ایک', 'مثال', 'ہے'] 4


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

AI, search! ['ai', ',', 'search', '!']
I can't log in. ['i', 'can', "'", 't', 'log', 'in', '.']

Observations:
1. Whitespace splitting keeps punctuation attached to words
   ('search!' stays one token), while the word-like regex drops
   punctuation, splits the contraction "can't" into 'can' and
   't', and splits the email address into separate parts.
2. The model tokenizer uses subword units and lowercases, so its
   token list and count differ from both methods above.
Normalization risk: lowercasing or stripping punctuation before
search can damage case-sensitive identifiers (CS4944, file names,
codes) and can remove negation cues, so the original text is
always kept unchanged and any normalized copy is stored separately.
Note: the regex is a simple word-like extractor, not a complete
multilingual tokenizer; the Urdu result shows letter sequences
only and does not prove correct Urdu tokenization.

PART B - FAQ CHATBOT DEMO (TF-IDF)
You: How do I reset my password?
Bot: Use the recov